# Evaluacion de Shocks — Dataset Extendido

**Problema:** La deteccion de shocks en el notebook de reentrenamiento usaba `pct_change()` sobre valores ya normalizados (z-scores), donde las variaciones absolutas son pequenas y el umbral de 20% nunca se alcanza.

**Solucion:** Desnormalizar `produccion_t` con los parametros del scaler guardado, detectar shocks sobre la serie real, y recalcular MAE global / MAE shock / deterioro para los 3 modelos.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

with open(ROOT / 'resultados/scaler_extendido.json') as f:
    scaler_info = json.load(f)

target_mean  = scaler_info['target_mean']
target_scale = scaler_info['target_scale']

print(f'Scaler target: mean={target_mean:.4f}  scale={target_scale:.4f}')

modelos = {
    'GE_ext':      ROOT / 'resultados/ge_ext/predicciones.csv',
    'GM_v3_ext':   ROOT / 'resultados/gm_v3_ext/predicciones.csv',
    'XGBoost_ext': ROOT / 'resultados/xgboost_ext/predicciones.csv',
}

for nombre, path in modelos.items():
    print(f'{nombre}: {path.exists()}  ({pd.read_csv(path).shape[0]} filas)')

Scaler target: mean=117.2424  scale=165.9714
GE_ext: True  (12 filas)
GM_v3_ext: True  (6 filas)
XGBoost_ext: True  (12 filas)


## Deteccion de shocks sobre produccion desnormalizada

Desnormalizar con `x_real = x_scaled * scale + mean`, luego `pct_change() > 20%`.

In [2]:
# Cargar serie completa para detectar shocks en escala real
df_master = pd.read_csv(ROOT / 'data/processed/master_escalado_extendido.csv',
                        parse_dates=['fecha_evento'])
df_master = df_master.sort_values('fecha_evento').reset_index(drop=True)

# Desnormalizar produccion
df_master['prod_real'] = df_master['produccion_t'] * target_scale + target_mean
df_master['variacion_pct'] = df_master['prod_real'].pct_change().abs() * 100

# Meses de test: ultimos 12
n_test = 12
df_test_master = df_master.iloc[-n_test:].copy()
fechas_shock = set(
    df_test_master[df_test_master['variacion_pct'] > 20]['fecha_evento'].dt.to_period('M')
)

print(f'Serie completa: {len(df_master)} meses')
print(f'Test: ultimos {n_test} meses')
print(f'Shocks detectados en test: {len(fechas_shock)} de {n_test} meses')
print(f'Meses de shock: {sorted(fechas_shock)}')
print()

# Detalle mes a mes
print(f'{"Fecha":<14} {"prod_real":>10} {"var_pct":>8} {"Shock":>6}')
print('-' * 42)
for _, r in df_test_master.iterrows():
    es = 'SI' if r['variacion_pct'] > 20 else ''
    print(f'{str(r["fecha_evento"].date()):<14} {r["prod_real"]:>10.4f} {r["variacion_pct"]:>7.1f}% {es:>6}')

Serie completa: 80 meses
Test: ultimos 12 meses
Shocks detectados en test: 8 de 12 meses
Meses de shock: [Period('2024-09', 'M'), Period('2024-10', 'M'), Period('2024-12', 'M'), Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M')]

Fecha           prod_real  var_pct  Shock
------------------------------------------
2024-09-01        -0.0106   229.9%     SI
2024-10-01        -0.0147    39.5%     SI
2024-11-01        -0.0142     3.6%       
2024-12-01         0.0059   141.3%     SI
2025-01-01        -0.0542  1021.4%     SI
2025-02-01        -0.0420    22.5%     SI
2025-03-01        -0.0500    19.0%       
2025-04-01        -0.0649    29.8%     SI
2025-05-01        -0.0860    32.6%     SI
2025-06-01        -0.1113    29.4%     SI
2025-07-01        -0.1019     8.4%       
2025-08-01        -0.1117     9.6%       


## MAE global y MAE en shocks por modelo

Comparar cada modelo con sus predicciones desnormalizadas contra la referencia original (2021-2025).

In [3]:
# Referencia original (2021-2025, del CLAUDE.md y notebooks ejecutados)
resultados_orig = {
    'GE_orig':   {'MAE': 0.0673, 'MAE_shock': 0.0688, 'det': '+2.3%'},
    'GM_v3_orig':{'MAE': 0.0645, 'MAE_shock': 0.0721, 'det': '+11.7%'},
    'XGB_orig':  {'MAE': 0.0471, 'MAE_shock': 0.0549, 'det': '+16.5%'},
}

resultados_ext = {}

for nombre, path in modelos.items():
    df = pd.read_csv(path, parse_dates=['fecha'])

    # Desnormalizar real y predicho
    df['real_desn'] = df['real'] * target_scale + target_mean
    df['pred_desn'] = df['predicho'] * target_scale + target_mean

    # Mapear meses de shock
    df['fecha_mes'] = df['fecha'].dt.to_period('M')
    df['es_shock'] = df['fecha_mes'].isin(fechas_shock)

    # MAE global (en escala normalizada, comparable con original)
    mae_global = mean_absolute_error(df['real'], df['predicho'])

    # MAE en shocks
    df_shock  = df[df['es_shock']]
    df_normal = df[~df['es_shock']]

    if len(df_shock) > 0:
        mae_shock  = mean_absolute_error(df_shock['real'], df_shock['predicho'])
        deterioro  = (mae_shock - mae_global) / mae_global * 100
    else:
        mae_shock  = float('nan')
        deterioro  = float('nan')

    mae_normal = mean_absolute_error(df_normal['real'], df_normal['predicho']) if len(df_normal) > 0 else float('nan')

    # MAE en escala real (toneladas)
    mae_real_global = mean_absolute_error(df['real_desn'], df['pred_desn'])
    mae_real_shock  = mean_absolute_error(df_shock['real_desn'], df_shock['pred_desn']) if len(df_shock) > 0 else float('nan')

    resultados_ext[nombre] = {
        'n_meses':     len(df),
        'n_shocks':    len(df_shock),
        'MAE':         mae_global,
        'MAE_shock':   mae_shock,
        'MAE_normal':  mae_normal,
        'deterioro':   deterioro,
        'MAE_real_t':  mae_real_global,
        'MAE_real_shock_t': mae_real_shock,
    }

    print(f'{nombre}:')
    print(f'  Meses test: {len(df)}  |  Shocks: {len(df_shock)}')
    print(f'  MAE (z-score):  global={mae_global:.4f}  shock={mae_shock:.4f}  normal={mae_normal:.4f}')
    print(f'  Deterioro:      {deterioro:+.1f}%')
    print(f'  MAE (tons):     global={mae_real_global:.2f}  shock={mae_real_shock:.2f}')
    print()

GE_ext:
  Meses test: 12  |  Shocks: 8
  MAE (z-score):  global=0.0035  shock=0.0044  normal=0.0018
  Deterioro:      +24.6%
  MAE (tons):     global=0.58  shock=0.73

GM_v3_ext:
  Meses test: 6  |  Shocks: 3
  MAE (z-score):  global=0.0505  shock=0.0490  normal=0.0519
  Deterioro:      -2.9%
  MAE (tons):     global=8.38  shock=8.14

XGBoost_ext:
  Meses test: 12  |  Shocks: 8
  MAE (z-score):  global=0.0004  shock=0.0004  normal=0.0005
  Deterioro:      -7.4%
  MAE (tons):     global=0.07  shock=0.06



## Tabla comparativa final: Original vs Extendido

In [4]:
print('=' * 85)
print('  COMPARATIVA FINAL: ORIGINAL (2021-2025) vs EXTENDIDO (2019-2025)')
print('=' * 85)
print(f'  {"Modelo":<18} {"MAE orig":>10} {"MAE ext":>10} {"Delta":>8} {"Det orig":>10} {"Det ext":>10}')
print('-' * 85)

pares = [
    ('GE sin NLP',  'GE_orig',    'GE_ext'),
    ('GM v3 NLP',   'GM_v3_orig', 'GM_v3_ext'),
    ('XGBoost',     'XGB_orig',   'XGBoost_ext'),
]

for label, key_orig, key_ext in pares:
    mae_o = resultados_orig[key_orig]['MAE']
    det_o = resultados_orig[key_orig]['det']
    r_ext = resultados_ext[key_ext]
    mae_e = r_ext['MAE']
    det_e = f'{r_ext["deterioro"]:+.1f}%' if not np.isnan(r_ext['deterioro']) else 'N/A'
    delta = (mae_e - mae_o) / mae_o * 100
    print(f'  {label:<18} {mae_o:>10.4f} {mae_e:>10.4f} {delta:>+7.1f}% {det_o:>10} {det_e:>10}')

print('=' * 85)

# Tabla de shocks detallada
print(f'\n  ANALISIS DE SHOCKS — {len(fechas_shock)} shocks de {n_test} meses en test')
print('=' * 75)
print(f'  {"Modelo":<18} {"n_test":>6} {"n_shock":>8} {"MAE_global":>10} {"MAE_shock":>10} {"MAE_norm":>10} {"Det":>8}')
print('-' * 75)

for label, _, key_ext in pares:
    r = resultados_ext[key_ext]
    ms = f'{r["MAE_shock"]:.4f}' if not np.isnan(r['MAE_shock']) else 'N/A'
    mn = f'{r["MAE_normal"]:.4f}' if not np.isnan(r['MAE_normal']) else 'N/A'
    dt = f'{r["deterioro"]:+.1f}%' if not np.isnan(r['deterioro']) else 'N/A'
    print(f'  {label:<18} {r["n_meses"]:>6} {r["n_shocks"]:>8} {r["MAE"]:>10.4f} {ms:>10} {mn:>10} {dt:>8}')

print('=' * 75)

# Referencia original
print(f'\n  Referencia original (n_train=40, n_test=10):')
for label, key_orig, _ in pares:
    r = resultados_orig[key_orig]
    print(f'    {label:<18} MAE={r["MAE"]:.4f}  MAE_shock={r["MAE_shock"]:.4f}  Det={r["det"]}')

# Guardar resultados corregidos
output = {
    'fechas_shock': [str(f) for f in sorted(fechas_shock)],
    'n_shocks': len(fechas_shock),
    'n_test': n_test,
    'resultados_extendido': {k: {kk: (vv if not isinstance(vv, float) or not np.isnan(vv) else None)
                                  for kk, vv in v.items()}
                              for k, v in resultados_ext.items()},
    'referencia_original': resultados_orig,
}
with open(ROOT / 'resultados/evaluacion_shocks_extendido.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f'\nGuardado en resultados/evaluacion_shocks_extendido.json')

  COMPARATIVA FINAL: ORIGINAL (2021-2025) vs EXTENDIDO (2019-2025)
  Modelo               MAE orig    MAE ext    Delta   Det orig    Det ext
-------------------------------------------------------------------------------------
  GE sin NLP             0.0673     0.0035   -94.8%      +2.3%     +24.6%
  GM v3 NLP              0.0645     0.0505   -21.7%     +11.7%      -2.9%
  XGBoost                0.0471     0.0004   -99.1%     +16.5%      -7.4%

  ANALISIS DE SHOCKS — 8 shocks de 12 meses en test
  Modelo             n_test  n_shock MAE_global  MAE_shock   MAE_norm      Det
---------------------------------------------------------------------------
  GE sin NLP             12        8     0.0035     0.0044     0.0018   +24.6%
  GM v3 NLP               6        3     0.0505     0.0490     0.0519    -2.9%
  XGBoost                12        8     0.0004     0.0004     0.0005    -7.4%

  Referencia original (n_train=40, n_test=10):
    GE sin NLP         MAE=0.0673  MAE_shock=0.0688  Det=+